In [1]:
from crewai import Agent, Task, Crew, LLM
from typing import Optional, Any

from crewai.tools import BaseTool

from src.data.jira_data_fetcher import JiraDataFetcher
from src.storage.data_manager import WeeklyDataManager
from src.storage.data_validator import DataValidator
from pydantic import Field

from src.agent_tools.storage import JiraStorageTools

In [2]:
# Initialize LLM
llm = LLM(model="ollama/llama3.2", base_url="http://localhost:11434")

In [4]:
# Create Storage Agent
data_storage_agent = Agent(
    role="Data Storage Specialist",
    goal="Efficiently manage and validate Jira data extraction and storage",
    backstory="Expert in data storage and validation with focus on data integrity",
    tools=[JiraStorageTools()],
    llm=llm
)

In [27]:
class WeeklyDataManagementTool(BaseTool):
    name: str = "Weekly Data Management"
    description: str = "Tool for storing and retrieving weekly Jira data with validation"

    def _run(self, data=None, week=None) -> str:
        """Required implementation of the abstract _run method"""
        self.data_manager = WeeklyDataManager()
        self.validator = DataValidator()
        if data:
            return self.save_data(data)
        elif week:
            return self.get_data(week)
        return "Please provide either data to save or week to retrieve"

    def save_data(self, data):
        """Save and validate weekly data"""
        valid, message = self.validator.validate_data(data)
        if not valid:
            return f"Data validation failed: {message}"
        
        week = self.data_manager.save_weekly_data(data)
        return f"Data saved successfully for week {week}"

    def get_data(self, week):
        """Retrieve weekly data"""
        data = self.data_manager.get_weekly_data(week)
        if data is None:
            return f"No data found for week {week}"
        return str(data)

In [28]:
# Create Agent
data_storage_agent = Agent(
    role="Data Storage Specialist",
    goal="Efficiently store and manage Jira data with validation",
    backstory="Expert in data storage and validation with focus on data integrity",
    tools=[WeeklyDataManagementTool()],
    llm=llm
)

In [31]:
# Create Task
store_data_task = Task(
    description="Store and validate weekly Jira data",
    expected_output="A confirmation message indicating successful data storage and validation status",
    agent=data_storage_agent
)

In [32]:
# Create and Run Crew
jira_crew = Crew(
    agents=[data_storage_agent],
    tasks=[store_data_task],
    verbose=True
)

# Execute
result = jira_crew.kickoff()


# Agent: Data Storage Specialist
## Task: Store and validate weekly Jira data


# Agent: Data Storage Specialist
## Using tool: Weekly Data Management
## Tool Input: 
"{\"validate\": true}"
## Tool Output: 
Error: the Action Input is not a valid key, value dictionary.


# Agent: Data Storage Specialist
## Using tool: Weekly Data Management
## Tool Input: 
"{\"data\": \"{'Jira Issues': {'Issue 1': 'Description 1', 'Issue 2': 'Description 2'}\", \"validate\": true}"
## Tool Output: 
Error: the Action Input is not a valid key, value dictionary.


# Agent: Data Storage Specialist
## Using tool: Weekly Data Management
## Tool Input: 
"{\"data\": \"{\\\"Jira Issues\\\": {\\\"Issue 1\\\": \\\"Description 1\\\", \\\"Issue 2\\\": \\\"Description 2\\\"}, \\\"validate\\\": true}\", \"format\": \"json\"}"
## Tool Output: 

I encountered an error while trying to use the tool. This was the error: "WeeklyDataManagementTool" object has no field "data_manager".
 Tool Weekly Data Management accepts thes